In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import polars as pl  # 确保导入 polars
from pathlib import Path
from tqdm.auto import tqdm
import os
import pickle
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, KFold
from torch.utils.data import Dataset, DataLoader, Sampler
import glob
import random
import gc
import time
import math
import traceback

# ============================================================================
# CONFIG (保持高性能配置)
# ============================================================================
class Config:
    IS_TRAINING = True
    
    SEED = 2025
    SEED_LIST = [2025]
    N_FOLDS = 5
    BATCH_SIZE = 256
    EPOCHS = 500
    PATIENCE = 30
    LEARNING_RATE = 1e-3
    
    WINDOW_SIZE = 10
    HIDDEN_DIM = 192
    MAX_FUTURE_HORIZON = 60 # 保持 60 帧预测
    
    N_HEADS = 8
    N_LAYERS = 1
    DROPOUT = 0.15
    HEAD_LAYERS = 2
    
    DATA_DIR = Path("nfl-big-data-bowl-2026-prediction")
    OUTPUT_DIR = Path("training_output") 
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    FIELD_X_MIN, FIELD_X_MAX = 0.0, 120.0
    FIELD_Y_MIN, FIELD_Y_MAX = 0.0, 53.3
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_AMP = True if DEVICE.type == 'cuda' else False
    
    USE_Y_FLIP = True
    Y_FLIP_PROB = 0.5

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# ============================================================================
# 辅助函数
# ============================================================================
def safe_div(a, b, eps=1e-6):
    return a / (b + eps)

def wrap_angle_deg(s):
    return ((s + 180.0) % 360.0) - 180.0

def unify_left_direction(df: pd.DataFrame) -> pd.DataFrame:
    if 'play_direction' not in df.columns: return df
    df = df.copy()
    right = df['play_direction'].eq('right')
    if 'x' in df.columns: df.loc[right, 'x'] = Config.FIELD_X_MAX - df.loc[right, 'x']
    if 'y' in df.columns: df.loc[right, 'y'] = Config.FIELD_Y_MAX - df.loc[right, 'y']
    for col in ('dir', 'o'):
        if col in df.columns: df.loc[right, col] = (df.loc[right, col] + 180.0) % 360.0
    if 'ball_land_x' in df.columns: df.loc[right, 'ball_land_x'] = Config.FIELD_X_MAX - df.loc[right, 'ball_land_x']
    if 'ball_land_y' in df.columns: df.loc[right, 'ball_land_y'] = Config.FIELD_Y_MAX - df.loc[right, 'ball_land_y']
    return df

def build_play_direction_map(df_in: pd.DataFrame) -> pd.Series:
    return df_in[['game_id', 'play_id', 'play_direction']].drop_duplicates().set_index(['game_id', 'play_id'])['play_direction']

def apply_direction_to_df(df: pd.DataFrame, dir_map: pd.Series) -> pd.DataFrame:
    if 'play_direction' not in df.columns:
        dir_df = dir_map.reset_index()
        df = df.merge(dir_df, on=['game_id', 'play_id'], how='left', validate='many_to_one')
    return unify_left_direction(df)

def load_and_merge_play_features(df, data_dir, is_training=True):
    possible_paths = [data_dir / "train" / "plays.csv", data_dir / "test" / "plays.csv", data_dir / "plays.csv"]
    plays_path = None
    for p in possible_paths:
        if p.exists(): plays_path = p; break
    
    if plays_path is None: return df
    
    plays = pd.read_csv(plays_path)
    use_cols = ['game_id', 'play_id', 'quarter', 'down', 'yardsToGo', 'yardlineNumber', 'absoluteYardlineNumber', 'gameClock', 'preSnapHomeScore', 'preSnapVisitorScore', 'possessionTeam', 'defensiveTeam']
    use_cols = [c for c in use_cols if c in plays.columns]
    plays = plays[use_cols]
    
    if 'gameClock' in plays.columns:
        def parse_clock(x):
            if pd.isna(x): return 0.0
            if isinstance(x, str):
                try:
                    if ':' in x: m, s = map(int, x.split(':')); return m * 60 + s
                except: pass
            return 0.0
        plays['gameClock_sec'] = plays['gameClock'].apply(parse_clock)
        plays = plays.drop(columns=['gameClock'])
        
    df = df.merge(plays, on=['game_id', 'play_id'], how='left')
    return df

# ============================================================================
# Feature Engineering
# ============================================================================
class FeatureEngineer:
    def __init__(self):
        self.gcols = ["game_id", "play_id", "nfl_id"]

    def _height_to_feet(self, height_str):
        try:
            ft, inches = map(int, str(height_str).split("-"))
            return ft + inches / 12
        except Exception:
            return 6.0
        
    def _mirror_angle(self, df: pd.DataFrame, cols: list):
        for col in cols:
            df[col] = (450 - df[col]) % 360
        return df
    
    def transform(self, df: pd.DataFrame):
        df = df.copy().sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
        
        # 1. Physical
        if "player_height" in df.columns:
            df["player_height_ft"] = df["player_height"].apply(self._height_to_feet)
            height_in = df["player_height_ft"] * 12
            df["bmi"] = (df["player_weight"] / (height_in**2 + 1e-6)) * 703
        else:
             df["player_height_ft"] = 6.0
             df["bmi"] = 30.0

        # 2. Kinematics
        df = self._mirror_angle(df, ["dir", "o"])
        dir_rad = np.deg2rad(df["dir"].fillna(0))
        o_rad = np.deg2rad(df["o"].fillna(0))
        df["vx"] = df["s"] * np.cos(dir_rad)
        df["vy"] = df["s"] * np.sin(dir_rad)
        df["ax"] = df["a"] * np.cos(dir_rad)
        df["ay"] = df["a"] * np.sin(dir_rad)
        df["ox"] = np.cos(o_rad)
        df["oy"] = np.sin(o_rad)
        
        df["momentum_x"] = df["vx"] * df["player_weight"]
        df["momentum_y"] = df["vy"] * df["player_weight"]
        df["momentum"] = df["s"] * df["player_weight"]
        df["kinetic_energy"] = 0.5 * df["player_weight"] * (df["s"]**2)
        df["orient_align"] = np.cos(dir_rad - o_rad)
        df["orient_perp"] = np.sin(dir_rad - o_rad)

        # 3. Field
        df["dist_sideline"] = np.minimum(df["y"], Config.FIELD_Y_MAX - df["y"])
        df["dist_endzone"] = df["x"] 
        df["dist_center"] = np.sqrt((df["x"] - Config.FIELD_X_MAX/2)**2 + (df["y"] - Config.FIELD_Y_MAX/2)**2)
        
        # 4. Ball
        if "ball_land_x" in df.columns:
            dx_ball = df["ball_land_x"] - df["x"]
            dy_ball = df["ball_land_y"] - df["y"]
            dist_ball = np.sqrt(dx_ball**2 + dy_ball**2)
            df["dist_to_ball"] = dist_ball
            df["ball_dx"] = dx_ball
            df["ball_dy"] = dy_ball
            df["dir_to_ball_x"] = safe_div(dx_ball, dist_ball)
            df["dir_to_ball_y"] = safe_div(dy_ball, dist_ball)
            df["course_to_ball_align"] = (df["vx"] * df["dir_to_ball_x"] + df["vy"] * df["dir_to_ball_y"]) / (df["s"] + 1e-6)
            df["closing_speed_ball"] = df["vx"] * df["dir_to_ball_x"] + df["vy"] * df["dir_to_ball_y"]
            df["time_to_ball"] = np.where(df["closing_speed_ball"] > 0.1, dist_ball / df["closing_speed_ball"], 10.0).clip(0, 10)
        else:
            cols_ball = ["dist_to_ball", "ball_dx", "ball_dy", "dir_to_ball_x", "dir_to_ball_y", "course_to_ball_align", "closing_speed_ball", "time_to_ball"]
            for c in cols_ball: df[c] = 0.0

        # 5. Role
        df["is_offense"] = (df["player_side"] == "Offense").astype(np.float32)
        df["is_defense"] = (df["player_side"] == "Defense").astype(np.float32)
        if "player_role" in df.columns:
            df["is_passer"] = (df["player_role"] == "Passer").astype(np.float32)
            df["is_receiver"] = (df["player_role"] == "Targeted Receiver").astype(np.float32)
        else:
            df["is_passer"] = 0.0
            df["is_receiver"] = 0.0
        
        if "player_role" in df.columns:
            rec_df = df[df['player_role'] == 'Targeted Receiver'][['game_id', 'play_id', 'frame_id', 'x', 'y']]
            rec_df = rec_df.rename(columns={'x': 'rx', 'y': 'ry'})
            df = df.merge(rec_df, on=['game_id', 'play_id', 'frame_id'], how='left')
            df['rx'] = df['rx'].fillna(df['x'])
            df['ry'] = df['ry'].fillna(df['y'])
            df['diff_receiver_x'] = df['x'] - df['rx']
            df['diff_receiver_y'] = df['y'] - df['ry']
        else:
            df['diff_receiver_x'] = 0.0
            df['diff_receiver_y'] = 0.0

        # 6. Temporal
        grp = df.groupby(self.gcols)
        df["dvx"] = grp["vx"].diff().fillna(0)
        df["dvy"] = grp["vy"].diff().fillna(0)
        df["d_dir"] = grp["dir"].diff().fillna(0)
        df["d_dir"] = wrap_angle_deg(df["d_dir"])
        df["d_acc"] = grp["a"].diff().fillna(0)
        df["time_seconds"] = df["frame_id"] / 10.0

        feature_cols = [
            "x", "y", "s", "a", "vx", "vy", "ax", "ay", "ox", "oy",
            "player_height_ft", "player_weight", "bmi",
            "momentum_x", "momentum_y", "momentum", "kinetic_energy",
            "orient_align", "orient_perp",
            "dist_sideline", "dist_endzone", "dist_center",
            "dist_to_ball", "ball_dx", "ball_dy", "dir_to_ball_x", "dir_to_ball_y",
            "course_to_ball_align", "closing_speed_ball", "time_to_ball",
            "is_offense", "is_defense", "is_passer", "is_receiver", 
            "diff_receiver_x", "diff_receiver_y",
            "dvx", "dvy", "d_dir", "d_acc", "time_seconds"
        ]
        final_cols = [c for c in feature_cols if c in df.columns]
        return df, final_cols
    
def feature_engineering(input_df, output_df=None, test_template=None, is_training=True):
    dir_map = build_play_direction_map(input_df)
    
    # 1. 统一 Input
    input_df = apply_direction_to_df(input_df, dir_map)
    
    # 2. 统一 Output (训练模式) - 必须修复这里！
    if is_training and output_df is not None:
        output_df = apply_direction_to_df(output_df, dir_map)
    
    # 3. 统一 Test Template (推理模式)
    if not is_training and test_template is not None:
        if 'play_direction' not in test_template.columns:
            dir_df = dir_map.reset_index()
            test_template = test_template.merge(dir_df, on=['game_id', 'play_id'], how='left', validate='many_to_one')
        test_template = apply_direction_to_df(test_template, dir_map)
    
    fe = FeatureEngineer()
    input_df, feature_cols = fe.transform(input_df)

    if is_training:
        return input_df, output_df, feature_cols
    else:
        return input_df, test_template, feature_cols

# ============================================================================
# Sequence Creation
# ============================================================================
def create_sequence(input_df, output_df=None, test_template=None, is_training=True, window_size=10, feature_cols=None):
    input_df = input_df.sort_values(['game_id', 'play_id', 'nfl_id', 'frame_id'])
    input_df[feature_cols] = input_df[feature_cols].ffill().fillna(0.0)
    
    keep_cols = ['game_id', 'play_id', 'nfl_id', 'frame_id'] + feature_cols
    data_arr = input_df[keep_cols].values
    col_idx = {c: i for i, c in enumerate(keep_cols)}
    feat_indices = [col_idx[c] for c in feature_cols]
    
    # Cache
    player_seq_cache = {}
    grp_indices = input_df.groupby(['game_id', 'play_id', 'nfl_id']).indices
    
    iterator = grp_indices.items()
    if is_training: iterator = tqdm(iterator, desc="Caching")

    for key, idxs in iterator:
        if len(idxs) > window_size:
            valid_idxs = idxs[-window_size:]
            seq = data_arr[valid_idxs][:, feat_indices]
        else:
            valid_idxs = idxs
            raw_seq = data_arr[valid_idxs][:, feat_indices]
            pad_len = window_size - len(raw_seq)
            seq = np.pad(raw_seq, ((pad_len, 0), (0, 0)), mode='constant', constant_values=0)
        player_seq_cache[key] = seq.astype(np.float32)

    task_df = output_df if is_training else test_template
    unique_plays = task_df[['game_id', 'play_id']].drop_duplicates()
    
    valid_play_outputs = {}
    if is_training:
        out_grps = output_df.groupby(['game_id', 'play_id'])
        for key, grp in out_grps:
            if grp['frame_id'].nunique() <= Config.MAX_FUTURE_HORIZON:
                valid_play_outputs[key] = grp
    
    play_to_players = input_df.groupby(['game_id', 'play_id'])['nfl_id'].unique().to_dict()
    play_to_dir = input_df.groupby(['game_id', 'play_id'])['play_direction'].first().to_dict()

    sequences = []
    targets_dx = []
    targets_dy = []
    targets_frame_ids = []
    sequence_ids = [] 
    game_ids_list = []
    
    idx_x = feature_cols.index('x')
    idx_y = feature_cols.index('y')

    iter_rows = unique_plays.iterrows()
    if is_training: iter_rows = tqdm(iter_rows, total=len(unique_plays), desc="Assembling")

    for _, row in iter_rows:
        gid, pid = row['game_id'], row['play_id']
        key = (gid, pid)

        if key not in play_to_players: continue
        all_player_ids = play_to_players[key]
        
        if is_training:
            if key not in valid_play_outputs: continue 
            play_output = valid_play_outputs[key]
            out_ids = play_output['nfl_id'].unique()
            target_nfl_ids = [pid for pid in out_ids if pid in all_player_ids]
            if not target_nfl_ids: continue
        else:
            target_nfl_ids = task_df[(task_df['game_id'] == gid) & (task_df['play_id'] == pid)]['nfl_id'].unique()

        play_dir_val = play_to_dir.get(key, 'left')

        for target_id in target_nfl_ids:
            target_key = (gid, pid, target_id)
            if target_key not in player_seq_cache: continue

            input_arrays = [player_seq_cache[target_key]]
            for neighbor_id in all_player_ids:
                if neighbor_id != target_id:
                    nb_key = (gid, pid, neighbor_id)
                    if nb_key in player_seq_cache:
                        input_arrays.append(player_seq_cache[nb_key])
            
            seq_tensor = np.stack(input_arrays, axis=1)
            sequences.append(seq_tensor)
            game_ids_list.append(gid)
            
            sequence_ids.append({
                'game_id': gid,
                'play_id': pid,
                'target_nfl_id': target_id,
                'n_valid_players': len(input_arrays),
                'play_direction': play_dir_val,
                'last_x': float(seq_tensor[-1, 0, idx_x]),
                'last_y': float(seq_tensor[-1, 0, idx_y])
            })
            
            if is_training:
                p_out = play_output[play_output['nfl_id'] == target_id]
                if not p_out.index.is_monotonic_increasing: p_out = p_out.sort_values('frame_id')
                
                last_x = seq_tensor[-1, 0, idx_x]
                last_y = seq_tensor[-1, 0, idx_y]
                
                dx = p_out['x'].values - last_x
                dy = p_out['y'].values - last_y
                fids = p_out['frame_id'].values
                
                targets_dx.append(dx.astype(np.float32))
                targets_dy.append(dy.astype(np.float32))
                targets_frame_ids.append(fids)

    if is_training:
        return sequences, targets_dx, targets_dy, targets_frame_ids, sequence_ids, np.array(game_ids_list)
    return sequences, sequence_ids

# ============================================================================
# Dataset & Sampler
# ============================================================================
class NFLDataset(Dataset):
    def __init__(self, sequences, targets_dx, targets_dy, targets_frame_ids, sequence_ids):
        self.sequences = sequences
        self.targets_dx = targets_dx
        self.targets_dy = targets_dy
        self.frame_ids = targets_frame_ids
        self.sequence_ids = sequence_ids
        self.player_counts = np.array([s['n_valid_players'] for s in sequence_ids])
        self.frame_counts = np.array([len(x) for x in targets_frame_ids]) if targets_dx else None

    def __len__(self): return len(self.sequences)
    
    def __getitem__(self, idx):
        item = {
            'sequence': self.sequences[idx],
            'player_count': self.player_counts[idx],
            'play_info': self.sequence_ids[idx]
        }
        if self.targets_dx: 
            item.update({
                'target_dx': self.targets_dx[idx],
                'target_dy': self.targets_dy[idx],
                'frame_ids': self.frame_ids[idx],
                'frame_count': self.frame_counts[idx]
            })
        return item

class GroupAwareSimilarSizeBatchSampler(Sampler):
    def __init__(self, player_counts, groups, batch_size, shuffle=True):
        self.player_counts = player_counts; self.batch_size = batch_size; self.shuffle = shuffle; self.indices = np.arange(len(player_counts))
    def __iter__(self):
        sorted_order = np.argsort(self.player_counts); indices = self.indices[sorted_order]
        if self.shuffle:
            chunk_size = 256; chunks = [indices[i:i+chunk_size] for i in range(0, len(indices), chunk_size)]
            for chunk in chunks: np.random.shuffle(chunk)
            indices = np.concatenate(chunks)
        batches = [indices[i:i+self.batch_size] for i in range(0, len(indices), self.batch_size)]
        if self.shuffle: np.random.shuffle(batches)
        for batch in batches: yield batch
    def __len__(self): return (len(self.indices) + self.batch_size - 1) // self.batch_size

def dynamic_collate_fn(batch, max_horizon=Config.MAX_FUTURE_HORIZON):
    B = len(batch); is_training = "target_dx" in batch[0]
    P_max = max(b['player_count'] for b in batch); T, _, F = batch[0]['sequence'].shape
    sequences = torch.zeros(B, T, P_max, F)
    player_masks = torch.zeros(B, P_max, dtype=torch.bool)
    
    if is_training:
        targets_dx = torch.zeros(B, max_horizon)
        targets_dy = torch.zeros(B, max_horizon)
        time_masks = torch.zeros(B, max_horizon, dtype=torch.bool)
        frame_ids = torch.zeros(B, max_horizon, dtype=torch.long)
    
    for i, sample in enumerate(batch):
        n_players = sample['player_count']
        sequences[i, :, :n_players, :] = torch.FloatTensor(sample['sequence'])
        player_masks[i, :n_players] = True
        if is_training:
            n_frames = sample['frame_count']
            targets_dx[i, :n_frames] = torch.FloatTensor(sample['target_dx'])
            targets_dy[i, :n_frames] = torch.FloatTensor(sample['target_dy'])
            time_masks[i, :n_frames] = True
            frame_ids[i, :n_frames] = torch.LongTensor(sample['frame_ids'])

    result = {'sequences': sequences, 'player_masks': player_masks, 'Meta': [b['play_info'] for b in batch]}
    if is_training:
        result.update({'targets_dx': targets_dx, 'targets_dy': targets_dy, 'time_masks': time_masks, 'frame_ids': frame_ids})
    return result

# ============================================================================
# Model & Components
# ============================================================================
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0. or not self.training: return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        return x.div(keep_prob) * random_tensor.floor()

class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.1):
        super().__init__()
        layers = []
        if num_layers <= 1: layers.append(nn.Linear(input_dim, output_dim))
        else:
            layers.append(nn.Linear(input_dim, hidden_dim)); layers.append(nn.LayerNorm(hidden_dim)); layers.append(nn.GELU()); layers.append(nn.Dropout(dropout))
            for _ in range(num_layers - 2):
                layers.append(nn.Linear(hidden_dim, hidden_dim)); layers.append(nn.LayerNorm(hidden_dim)); layers.append(nn.GELU()); layers.append(nn.Dropout(dropout))
            layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class SpatioTemporalBlock(nn.Module):
    def __init__(self, hidden_dim, n_heads, dropout=0.1, drop_path=0.0, spatial_first=False):
        super().__init__()
        self.spatial_first = spatial_first
        self.norm_t = nn.LayerNorm(hidden_dim); self.attn_t = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=n_heads, dropout=dropout, batch_first=True); self.drop_path_t = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm_s = nn.LayerNorm(hidden_dim); self.attn_s = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=n_heads, dropout=dropout, batch_first=True); self.drop_path_s = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm_ffn = nn.LayerNorm(hidden_dim); self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout)); self.drop_path_ffn = DropPath(drop_path) if drop_path > 0. else nn.Identity()
    def _temporal_attn(self, x):
        B, T, P, D = x.shape; x_t = x.permute(0, 2, 1, 3).reshape(B * P, T, D); shortcut = x_t
        x_t = self.norm_t(x_t); x_t, _ = self.attn_t(x_t, x_t, x_t); x_t = shortcut + self.drop_path_t(x_t)
        return x_t.reshape(B, P, T, D).permute(0, 2, 1, 3)
    def _spatial_attn(self, x, key_padding_mask):
        B, T, P, D = x.shape; x_s = x.reshape(B * T, P, D)
        mask_s = key_padding_mask.unsqueeze(1).repeat(1, T, 1).reshape(B * T, P) if key_padding_mask is not None else None
        shortcut = x_s; x_s = self.norm_s(x_s); x_s, _ = self.attn_s(x_s, x_s, x_s, key_padding_mask=mask_s); x_s = shortcut + self.drop_path_s(x_s)
        return x_s.reshape(B, T, P, D)
    def forward(self, x, key_padding_mask=None):
        if self.spatial_first: x = self._spatial_attn(x, key_padding_mask); x = self._temporal_attn(x)
        else: x = self._temporal_attn(x); x = self._spatial_attn(x, key_padding_mask)
        shortcut = x; x = self.norm_ffn(x); x = self.ffn(x); x = shortcut + self.drop_path_ffn(x)
        return x

class STTransformer(nn.Module):
    def __init__(self, input_dim, hidden_dim=160, n_heads=4, n_layers=2, dropout=0.2, drop_path_rate=0.1, horizon=55, window_size=10, mlp_layers=2, spatial_first=False, **kwargs):
        super().__init__()
        self.horizon = horizon; self.embedding = nn.Linear(input_dim, hidden_dim); self.temp_pos_embed = nn.Parameter(torch.zeros(1, window_size, 1, hidden_dim)); self.embed_dropout = nn.Dropout(dropout)      
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, n_layers)] 
        self.blocks = nn.ModuleList([SpatioTemporalBlock(hidden_dim, n_heads, dropout, dpr[i], spatial_first) for i in range(n_layers)])   
        self.final_norm = nn.LayerNorm(hidden_dim); self.head = MLPHead(hidden_dim, hidden_dim, horizon * 2, mlp_layers, dropout)
        nn.init.trunc_normal_(self.temp_pos_embed, std=0.02)
    def forward(self, x, player_mask=None, **kwargs):
        B, T, P, F = x.shape
        key_padding_mask = ~player_mask if player_mask is not None else None
        x = self.embedding(x) + self.temp_pos_embed[:, :T, :, :]; x = self.embed_dropout(x)    
        for block in self.blocks: x = block(x, key_padding_mask=key_padding_mask)   
        x = self.final_norm(x); target_feat = x[:, -1, 0, :]
        out = self.head(target_feat).reshape(B, self.horizon, 2)
        return torch.cumsum(out, dim=1) 

# ============================================================================
# Loss & Training Components
# ============================================================================
class TemporalHuber(nn.Module):
    def __init__(self, delta=0.5, time_decay=0.03, lam_smooth=0.01):
        super().__init__(); self.delta, self.time_decay, self.lam_smooth = delta, time_decay, lam_smooth
    def forward(self, pred, target, time_mask):
        err = pred - target; abs_err = torch.abs(err)
        huber = torch.where(abs_err <= self.delta, 0.5 * err * err, self.delta * (abs_err - 0.5 * self.delta))
        weights = time_mask.unsqueeze(-1).float()
        if self.time_decay > 0: weights = weights * torch.exp(-self.time_decay * torch.arange(pred.size(1), device=pred.device).float()).view(1, pred.size(1), 1)
        loss_weighted = huber * weights
        main_loss = loss_weighted.sum() / (weights.sum() + 1e-8)
        if self.lam_smooth > 0 and pred.size(1) > 2:
            d1 = pred[:, 1:, :] - pred[:, :-1, :]; d2 = d1[:, 1:, :] - d1[:, :-1, :]
            smooth_loss = ((d2 ** 2) * weights[:, 2:, :]).sum() / (weights[:, 2:, :].sum() + 1e-8)
        else: smooth_loss = pred.new_tensor(0.0)
        return main_loss + self.lam_smooth * smooth_loss

# 修复：FutureWarning logic for torch.cuda.amp.GradScaler
if Config.USE_AMP: 
    try:
        # Pytorch 2.4+
        amp_scaler = torch.amp.GradScaler('cuda')
    except AttributeError:
        # Older versions
        amp_scaler = torch.cuda.amp.GradScaler()
else:
    class MockScaler:
        def scale(self, loss): return loss
        def step(self, optimizer): optimizer.step()
        def update(self): pass
        def unscale_(self, opt): pass
    amp_scaler = MockScaler()

def train_one_epoch(model, loader, optimizer, criterion, device, y_flip_params):
    model.train(); total_loss = 0
    for batch in loader:
        if Config.USE_Y_FLIP and random.random() < Config.Y_FLIP_PROB: batch = train_batch_y_flip(batch, y_flip_params)
        X = batch['sequences'].to(device); p_mask = batch['player_masks'].to(device)
        t_dx = batch['targets_dx'].to(device); t_dy = batch['targets_dy'].to(device)
        time_mask = batch['time_masks'].to(device)
        
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=Config.USE_AMP):
            pred = model(X, player_mask=p_mask)
            targets = torch.stack([t_dx, t_dy], dim=-1)
            loss = criterion(pred, targets, time_mask)
        
        if torch.isnan(loss): continue
        if Config.USE_AMP: amp_scaler.scale(loss).backward(); amp_scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); amp_scaler.step(optimizer); amp_scaler.update()
        else: loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def compute_rmse(pred, target, time_mask):
    sq_dist = ((pred - target) ** 2).sum(dim=-1)
    valid_sq_dist = sq_dist * time_mask 
    total_dist = valid_sq_dist.sum()
    count = time_mask.sum()
    return torch.sqrt(total_dist / (2 * count + 1e-8)).item()

def validate(model, loader, criterion, device):
    model.eval(); total_loss = 0; val_rmses = []
    with torch.no_grad():
        for batch in loader:
            X = batch['sequences'].to(device); p_mask = batch['player_masks'].to(device)
            t_dx = batch['targets_dx'].to(device); t_dy = batch['targets_dy'].to(device)
            time_mask = batch['time_masks'].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=Config.USE_AMP):
                pred = model(X, player_mask=p_mask)
                targets = torch.stack([t_dx, t_dy], dim=-1)
                loss = criterion(pred, targets, time_mask)
            total_loss += loss.item()
            val_rmses.append(compute_rmse(pred, targets, time_mask))
    return total_loss / max(1, len(loader)), np.mean(val_rmses)

# ============================================================================
# TTA & Prediction Components
# ============================================================================
def compute_y_flip_params(feature_cols, scaler):
    means = scaler.mean_; stds = scaler.scale_
    NEGATE_FEATURES = ['vy', 'ay', 'oy', 'momentum_y', 'ball_dy', 'dir_to_ball_y', 'diff_receiver_y', 'dvy', 'd_dir', 'orient_perp']
    OFFSET_Y_FEATURES = ['y']
    negate_indices = [i for i, c in enumerate(feature_cols) if c in NEGATE_FEATURES]
    offset_y_indices = [i for i, c in enumerate(feature_cols) if c in OFFSET_Y_FEATURES]
    offset_y_values = [(Config.FIELD_Y_MAX - 2 * means[i]) / stds[i] for i in offset_y_indices]
    return {'negate_indices': negate_indices, 'offset_y_indices': offset_y_indices, 'offset_y_values': torch.tensor(offset_y_values, dtype=torch.float32)}

def train_batch_y_flip(batch, y_flip_params):
    sequences = batch['sequences']; targets_dy = batch['targets_dy']
    negate_indices = y_flip_params['negate_indices']
    offset_y_indices = y_flip_params['offset_y_indices']
    offset_y_values = y_flip_params['offset_y_values']
    if len(negate_indices) > 0: sequences[..., negate_indices] = -sequences[..., negate_indices]
    for i, idx in enumerate(offset_y_indices): sequences[..., idx] = offset_y_values[i] - sequences[..., idx]
    batch['targets_dy'] = -targets_dy
    return batch

def predict_with_y_flip_tta(model, seq_tensor, player_mask, y_flip_params, device, flip_weight=0.5):
    model.eval()
    negate_indices = y_flip_params['negate_indices']
    offset_y_indices = y_flip_params['offset_y_indices']
    offset_y_values = y_flip_params['offset_y_values'].to(device)
    with torch.no_grad():
        pred_original = model(seq_tensor, player_mask)
        seq_flipped = seq_tensor.clone()
        if len(negate_indices) > 0: seq_flipped[..., negate_indices] = -seq_flipped[..., negate_indices]
        for i, idx in enumerate(offset_y_indices): seq_flipped[..., idx] = offset_y_values[i] - seq_flipped[..., idx]
        pred_flipped = model(seq_flipped, player_mask)
        pred_flipped_restored = pred_flipped.clone()
        pred_flipped_restored[..., 1] = -pred_flipped[..., 1]
        pred_ensemble = (1 - flip_weight) * pred_original + flip_weight * pred_flipped_restored
    return pred_ensemble

def _collate_inference_batch(sequences):
    B = len(sequences)
    if B == 0: return None, None
    T, _, F = sequences[0].shape
    P_max = max([seq.shape[1] for seq in sequences])
    batch_seq = torch.zeros((B, T, P_max, F), dtype=torch.float32)
    batch_mask = torch.zeros((B, P_max), dtype=torch.bool)
    for i, seq in enumerate(sequences):
        p_count = seq.shape[1]
        batch_seq[i, :, :p_count, :] = torch.from_numpy(seq)
        batch_mask[i, :p_count] = True
    return batch_seq, batch_mask

def _predict_batch_inference(sequences, sequence_ids, models, scalers, feature_cols, device, y_flip_list):
    if len(sequences) == 0: return {}
    batch_seq, batch_mask = _collate_inference_batch(sequences)
    B, T, P, F = batch_seq.shape
    ensemble_preds = np.zeros((B, Config.MAX_FUTURE_HORIZON, 2), dtype=np.float32)
    valid_models_count = 0
    
    for model_idx, (model, scaler) in enumerate(zip(models, scalers)):
        try:
            flat_input = batch_seq.reshape(-1, F).numpy()
            scaled_flat = scaler.transform(flat_input)
            scaled_seq = scaled_flat.reshape(B, T, P, F)
            seq_tensor = torch.tensor(scaled_seq, dtype=torch.float32).to(device)
            mask_tensor = batch_mask.to(device)
            
            if model_idx < len(y_flip_list):
                pred_tensor = predict_with_y_flip_tta(model, seq_tensor, mask_tensor, y_flip_list[model_idx], device)
            else:
                pred_tensor = model(seq_tensor, mask_tensor)
            
            ensemble_preds += pred_tensor.cpu().numpy()
            valid_models_count += 1
        except Exception: continue
            
    if valid_models_count > 0: ensemble_preds /= valid_models_count
    
    results = {}
    for i, meta in enumerate(sequence_ids):
        nfl_id = meta['target_nfl_id']
        last_x = float(sequences[i][-1, 0, feature_cols.index('x')])
        last_y = float(sequences[i][-1, 0, feature_cols.index('y')])
        pred_dx = ensemble_preds[i, :, 0]
        pred_dy = ensemble_preds[i, :, 1]
        pred_x = np.clip(last_x + pred_dx, 0.0, 120.0)
        pred_y = np.clip(last_y + pred_dy, 0.0, 53.3)
        results[nfl_id] = np.stack([pred_x, pred_y], axis=-1)
    return results

# ============================================================================
# Training Pipeline
# ============================================================================
def load_weekly_data(week_num):
    input_df = pd.read_csv(f'{Config.DATA_DIR}/train/input_2023_w{week_num:02d}.csv')
    output_df = pd.read_csv(f'{Config.DATA_DIR}/train/output_2023_w{week_num:02d}.csv')
    return input_df, output_df

def load_all_train_data():
    print("Loading training data...")
    results = Parallel(n_jobs=4)(delayed(load_weekly_data)(i) for i in tqdm(range(1, 10)))
    input_dfs = [r[0] for r in results]; output_dfs = [r[1] for r in results]
    return pd.concat(input_dfs, ignore_index=True), pd.concat(output_dfs, ignore_index=True)

def run_training():
    print("🏆 Starting Training Pipeline 🏆")
    input_df, output_df = load_all_train_data()
    input_df, output_df, feature_cols = feature_engineering(input_df, output_df, is_training=True)
    sequences, targets_dx, targets_dy, targets_frame_ids, sequence_ids, game_ids_array = create_sequence(
        input_df, output_df, is_training=True, window_size=Config.WINDOW_SIZE, feature_cols=feature_cols
    )
    
    # Save Meta
    torch.save({'input_dim': sequences[0].shape[-1], 'feature_cols': feature_cols, 'config': Config}, Config.OUTPUT_DIR / "meta.pth")
    unique_groups = np.unique(game_ids_array)
    
    for seed in Config.SEED_LIST:
        print(f"\n🌱 Training Seed {seed}")
        set_seed(seed)
        SEED_DIR = Config.OUTPUT_DIR / f"seed_{seed}"; SEED_DIR.mkdir(exist_ok=True)
        GLOBAL_STATE_FILE = SEED_DIR / 'global_state.pth'
        
        models, scalers, y_flip_params_list = [], [], []
        kf = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=seed)
        
        for fold, (tr_idx_g, va_idx_g) in enumerate(kf.split(unique_groups), 1):
            train_groups, val_groups = unique_groups[tr_idx_g], unique_groups[va_idx_g]
            tr = np.where(np.isin(game_ids_array, train_groups))[0]
            va = np.where(np.isin(game_ids_array, val_groups))[0]
            
            # Prepare Data
            X_tr = [sequences[i] for i in tr]; X_va = [sequences[i] for i in va]
            dx_tr, dy_tr = [targets_dx[i] for i in tr], [targets_dy[i] for i in tr]
            dx_va, dy_va = [targets_dx[i] for i in va], [targets_dy[i] for i in va]
            meta_tr, meta_va = [sequence_ids[i] for i in tr], [sequence_ids[i] for i in va]
            fid_tr, fid_va = [targets_frame_ids[i] for i in tr], [targets_frame_ids[i] for i in va]

            # Scaler
            scaler = StandardScaler()
            X_tr_flat = np.concatenate([seq.reshape(-1, seq.shape[-1]) for seq in X_tr], axis=0)
            scaler.fit(X_tr_flat); del X_tr_flat
            scalers.append(scaler)
            y_flip_params = compute_y_flip_params(feature_cols, scaler)
            y_flip_params_list.append(y_flip_params)
            
            X_tr_s = [scaler.transform(seq.reshape(-1, seq.shape[-1])).reshape(seq.shape) for seq in X_tr]
            X_va_s = [scaler.transform(seq.reshape(-1, seq.shape[-1])).reshape(seq.shape) for seq in X_va]
            
            train_ds = NFLDataset(X_tr_s, dx_tr, dy_tr, fid_tr, meta_tr)
            valid_ds = NFLDataset(X_va_s, dx_va, dy_va, fid_va, meta_va)
            
            train_loader = DataLoader(train_ds, batch_sampler=GroupAwareSimilarSizeBatchSampler(train_ds.player_counts, None, Config.BATCH_SIZE), collate_fn=dynamic_collate_fn, num_workers=0)
            valid_loader = DataLoader(valid_ds, batch_sampler=GroupAwareSimilarSizeBatchSampler(valid_ds.player_counts, None, Config.BATCH_SIZE, shuffle=False), collate_fn=dynamic_collate_fn, num_workers=0)
            
            # Model & Train
            # 关键修复：显式传入 horizon 参数
            model = STTransformer(
                input_dim=len(feature_cols),
                horizon=Config.MAX_FUTURE_HORIZON 
            ).to(Config.DEVICE)

            optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
            criterion = TemporalHuber().to(Config.DEVICE)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, mode='min')
            
            best_rmse = float('inf'); patience = 0; best_state = None
            for epoch in range(Config.EPOCHS):
                t_loss = train_one_epoch(model, train_loader, optimizer, criterion, Config.DEVICE, y_flip_params)
                v_loss, v_rmse = validate(model, valid_loader, criterion, Config.DEVICE)
                scheduler.step(v_rmse)
                
                print(f"Epoch {epoch}: Train Loss {t_loss:.4f}, Val RMSE {v_rmse:.4f}")
                
                if v_rmse < best_rmse:
                    best_rmse = v_rmse; best_state = model.state_dict(); patience = 0
                else: patience += 1
                if patience >= Config.PATIENCE: break
            
            model.load_state_dict(best_state)
            models.append(model.cpu()) # Move to CPU for saving
            print(f"  Fold {fold} Best RMSE: {best_rmse:.5f}")
        
        # Save Global State per Seed
        torch.save({'models': models, 'scalers': scalers, 'y_flip_params_list': y_flip_params_list, 'feature_cols': feature_cols}, GLOBAL_STATE_FILE)
    print("Training Complete!")

# ============================================================================
# Global Variables & Load Logic
# ============================================================================
_models_loaded = False
_models = []
_scalers = []
_y_flip_params_list = []
_feature_cols = None
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_models():
    global _models_loaded, _models, _scalers, _y_flip_params_list, _feature_cols
    if _models_loaded: return

    CHECKPOINT_DIR = 'sttransformerMultiSeed1202' 
    
    print("Loading models (Multi-Seed Ensemble)...")
    try:
        total_models = 0
        for seed in Config.SEED_LIST:
            SEED_DIR = f'{CHECKPOINT_DIR}/seed_{seed}'
            state_file = f'{SEED_DIR}/global_state.pth'
            if not os.path.exists(state_file): continue
            
            state = torch.load(state_file, map_location=_device, weights_only=False)
            
            if _feature_cols is None: _feature_cols = state.get('feature_cols', None)
            
            for model in state['models']:
                model.to(_device).eval()
                _models.append(model)
                total_models += 1
            _scalers.extend(state['scalers'])
            if 'y_flip_params_list' in state: _y_flip_params_list.extend(state['y_flip_params_list'])
            
        _models_loaded = True
        print(f"Loaded {total_models} models.")
        
    except Exception as e:
        print(f"Error loading models: {e}")
        traceback.print_exc()

def _create_default_predictions(test_pd: pd.DataFrame) -> pl.DataFrame:
    df = pd.DataFrame({
        'x': [60.0] * len(test_pd),
        'y': [26.65] * len(test_pd)
    })
    return pl.from_pandas(df)

# ============================================================================
# Main Prediction Function
# ============================================================================
def predict(test: pl.DataFrame, test_input: pl.DataFrame) -> pl.DataFrame:
    try:
        if not _models_loaded: load_models()
        if not _models: return _create_default_predictions(test.to_pandas())
        
        test_pd = test.to_pandas()
        test_in_pd = test_input.to_pandas()
        
        test_in_pd = load_and_merge_play_features(test_in_pd, Config.DATA_DIR, is_training=False)
        if _feature_cols:
             for col in _feature_cols:
                if col not in test_in_pd.columns: test_in_pd[col] = 0.0
        
        test_input_features, test_template_u, _ = feature_engineering(
            test_in_pd, test_template=test_pd, is_training=False
        )
        
        sequences, sequence_ids = create_sequence(
            test_input_features, 
            test_template=test_template_u, 
            is_training=False,
            window_size=Config.WINDOW_SIZE,
            feature_cols=_feature_cols
        )
        
        if len(sequences) == 0: return _create_default_predictions(test_pd)
        
        player_predictions = _predict_batch_inference(
            sequences, sequence_ids, _models, _scalers, _feature_cols, _device, _y_flip_params_list
        )
        
        play_dir_lookup = {
            (meta['game_id'], meta['play_id']): meta['play_direction'] 
            for meta in sequence_ids
        }
        
        last_frames = test_in_pd.groupby(['game_id', 'play_id', 'nfl_id'])['frame_id'].max().to_dict()
        
        final_rows = []
        for _, row in test_pd.iterrows():
            gid, pid, nfl_id, fid = row['game_id'], row['play_id'], row['nfl_id'], row['frame_id']
            k = (gid, pid, nfl_id)
            pred_x, pred_y = 60.0, 26.65 # 默认值
            
            if k in player_predictions and k in last_frames:
                idx = int(fid - last_frames[k] - 1)
                
                if 0 <= idx < Config.MAX_FUTURE_HORIZON:
                    px, py = player_predictions[nfl_id][idx]
                    
                    if play_dir_lookup.get((gid, pid), 'left') == 'right':
                        px = Config.FIELD_X_MAX - px
                        py = Config.FIELD_Y_MAX - py
                    
                    pred_x, pred_y = px, py
            
            final_rows.append({'x': pred_x, 'y': pred_y})
            
        return pl.from_pandas(pd.DataFrame(final_rows))
        
    except Exception as e:
        print(f"CRITICAL ERROR in predict: {e}")
        traceback.print_exc()
        return _create_default_predictions(test.to_pandas())

if __name__ == "__main__":
    if Config.IS_TRAINING:
        run_training()
    else:
        import kaggle_evaluation.nfl_inference_server
        inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            inference_server.serve()
        else:
            inference_server.run_local_gateway(
                (Config.DATA_DIR,)
            )

/home/xxc/miniconda3/envs/my_project_env/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


🏆 Starting Training Pipeline 🏆
Loading training data...


  0%|          | 0/9 [00:00<?, ?it/s]

/home/xxc/miniconda3/envs/my_project_env/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/xxc/miniconda3/envs/my_project_env/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/xxc/miniconda3/envs/my_project_env/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.

Caching:   0%|          | 0/87473 [00:00<?, ?it/s]

Assembling:   0%|          | 0/7118 [00:00<?, ?it/s]


🌱 Training Seed 2025
Epoch 0: Train Loss 0.4875, Val RMSE 0.9168
Epoch 1: Train Loss 0.3176, Val RMSE 0.7853
Epoch 2: Train Loss 0.2789, Val RMSE 0.7640
Epoch 3: Train Loss 0.2680, Val RMSE 0.7207
Epoch 4: Train Loss 0.2423, Val RMSE 0.7089
Epoch 5: Train Loss 0.2333, Val RMSE 0.6867
Epoch 6: Train Loss 0.2247, Val RMSE 0.7280
Epoch 7: Train Loss 0.2195, Val RMSE 0.7103
Epoch 8: Train Loss 0.2178, Val RMSE 0.6531
Epoch 9: Train Loss 0.2116, Val RMSE 0.6432
Epoch 10: Train Loss 0.2059, Val RMSE 0.6527
Epoch 11: Train Loss 0.2021, Val RMSE 0.6791
Epoch 12: Train Loss 0.1986, Val RMSE 0.6591
Epoch 13: Train Loss 0.1997, Val RMSE 0.6223
Epoch 14: Train Loss 0.1967, Val RMSE 0.6706
Epoch 15: Train Loss 0.1935, Val RMSE 0.6226
Epoch 16: Train Loss 0.1914, Val RMSE 0.6389
Epoch 17: Train Loss 0.1899, Val RMSE 0.6054
Epoch 18: Train Loss 0.1837, Val RMSE 0.6251
Epoch 19: Train Loss 0.1845, Val RMSE 0.6309
Epoch 20: Train Loss 0.1848, Val RMSE 0.6129
Epoch 21: Train Loss 0.1854, Val RMSE 0.678